# Lab 06 solution

In [ ]:
import re
import time
from datetime import date
from io import StringIO
from pathlib import Path
from urllib import robotparser

import requests
import pandas as pd
from bs4 import BeautifulSoup

URL = "https://en.wikipedia.org/wiki/List_of_countries_by_exports"
HEADERS = {"User-Agent": "TradeCourse/1.0 (training exercise)"}
CACHE = Path("../../data/cache")
OUT = Path("output")
OUT.mkdir(exist_ok=True)

In [ ]:
robots = requests.get("https://en.wikipedia.org/robots.txt", headers=HEADERS, timeout=30)
rp = robotparser.RobotFileParser()
rp.parse(robots.text.splitlines())
print("Allowed:", rp.can_fetch(HEADERS["User-Agent"], URL))

Other checks: the site's **terms of use** (Wikimedia permits reuse under CC BY-SA with attribution); whether an **API or download** exists instead (the original World Bank data has an API, see Module 05); **server load** (one request, cached, is fine; thousands per minute is not); and whether the page contains **personal data** (it does not).

In [ ]:
def get_html(url):
    try:
        resp = requests.get(url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        print("Downloaded live page")
        return resp.text
    except requests.RequestException as e:
        print(f"Download failed ({e}); using cached copy")
        return (CACHE / "wikipedia_exports.html").read_text(encoding="utf-8")

html = get_html(URL)

In [ ]:
# check
assert "<table" in html
print("Task 2 OK")

In [ ]:
soup = BeautifulSoup(html, "html.parser")
table = None
for t in soup.find_all("table"):
    cap = t.find("caption")
    if cap and "Exports" in cap.get_text():
        table = t
        break

trs = table.find_all("tr")
headers = [th.get_text(" ", strip=True) for th in trs[0].find_all("th")]
rows = [[c.get_text(" ", strip=True) for c in tr.find_all(["th", "td"])] for tr in trs[1:]]
print(headers)
print(rows[:3])

In [ ]:
# check
assert len(headers) == 4
assert len(rows) > 150
print("Task 3 OK")

In [ ]:
def clean_text(text):
    return re.sub(r"\[.*?\]", "", text).strip()

exports = pd.DataFrame(
    [[clean_text(c) for c in r] for r in rows],
    columns=["country", "exports_usd_m", "year", "top_export"],
)
exports["exports_usd_m"] = exports["exports_usd_m"].str.replace(",", "").astype(float)
exports["year"] = exports["year"].astype(int)
exports["source"] = URL
exports["retrieved"] = date.today().isoformat()
exports.head()

In [ ]:
# check
assert exports["exports_usd_m"].dtype == float
assert exports["year"].dtype.kind == "i"
assert not exports["country"].str.contains(r"\[").any()
print("Task 4 OK")

In [ ]:
tables = pd.read_html(StringIO(html), match="Exports")
rh = tables[0]
print(len(rh), "rows via read_html vs", len(exports), "via BeautifulSoup")
print(rh.dtypes)
# read_html keeps footnote text in the headers and may read numbers as text or int;
# we would still need to rename columns and check types.

In [ ]:
print(exports["year"].value_counts().sort_index())
latest = exports["year"].max()
print((exports["year"] < latest).sum(), "countries with data older than", latest)

countries = pd.read_excel("../../data/countries.xlsx")
missing = sorted(set(countries["country_name"]) - set(exports["country"]))
print("Course economies not matched by name:", missing)

exports.to_csv(OUT / "web_exports.csv", index=False)

**Notes:** the table mixes reference years, so ranking countries directly compares different years. Figures combine goods **and** services, unlike our merchandise-only trade file. Country names follow Wikipedia conventions (for example `Vietnam` rather than `Viet Nam`), so joining to other sources needs a name-to-ISO3 mapping: see Module 07.